# 06. Ensemble Evaluation

Comparing individual models against the Ensemble approach (20% SARIMA, 50% XGBoost, 30% LSTM).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_absolute_percentage_error

df = pd.read_csv("../backend/data/processed/engineered_features.csv")
df["date"] = pd.to_datetime(df["date"])

features = ["price_lag_7", "price_lag_14", "price_lag_30", "rolling_mean_7", "rolling_mean_30", "rolling_std_7", "day_of_week", "month", "temperature", "rainfall", "ndvi_proxy"]

def evaluate_ensemble(commodity):
    data = df[df["commodity"] == commodity].sort_values("date")
    
    # We'll evaluate on the last 30 days
    test_data = data.iloc[-30:]
    y_test = test_data["price"].values
    
    # Load models
    sarima = joblib.load(f"../backend/models/saved/sarima_{commodity}.pkl")
    xgb = joblib.load(f"../backend/models/saved/xgboost_{commodity}.pkl")
    lstm = load_model(f"../backend/models/saved/lstm_{commodity}.h5")
    scaler = joblib.load(f"../backend/models/saved/scaler_{commodity}.pkl")
    
    # SARIMA Pred
    pred_sarima = sarima.get_forecast(steps=30).predicted_mean.values
    
    # XGB Pred
    pred_xgb = xgb.predict(test_data[features])
    
    # LSTM Pred
    # We need sequences for LSTM
    data_prices = data["price"].values.reshape(-1, 1)
    data_scaled = scaler.transform(data_prices)
    
    lstm_preds = []
    # Quick sequence creation for the last 30 days
    for i in range(len(data)-30, len(data)):
        seq = data_scaled[i-30:i].reshape(1, 30, 1)
        lstm_preds.append(lstm.predict(seq, verbose=0)[0][0])
    
    pred_lstm = scaler.inverse_transform(np.array(lstm_preds).reshape(-1, 1)).flatten()
    
    # Ensemble
    pred_ensemble = (0.2 * pred_sarima) + (0.5 * pred_xgb) + (0.3 * pred_lstm)
    
    # Metrics
    mape_sarima = mean_absolute_percentage_error(y_test, pred_sarima)
    mape_xgb = mean_absolute_percentage_error(y_test, pred_xgb)
    mape_lstm = mean_absolute_percentage_error(y_test, pred_lstm)
    mape_ens = mean_absolute_percentage_error(y_test, pred_ensemble)
    
    print(f"--- {commodity.capitalize()} MAPE ---")
    print(f"SARIMA:  {mape_sarima:.4f}")
    print(f"XGBoost: {mape_xgb:.4f}")
    print(f"LSTM:    {mape_lstm:.4f}")
    print(f"Ensemble:{mape_ens:.4f}")
    
    plt.figure(figsize=(10, 5))
    plt.plot(test_data["date"], y_test, label="Actual", linewidth=2)
    plt.plot(test_data["date"], pred_ensemble, label="Ensemble", linestyle="--", linewidth=2)
    plt.title(f"{commodity.capitalize()} Ensemble Forecast vs Actual")
    plt.legend()
    plt.savefig(f"06_ensemble_{commodity}.png")
    plt.close()

for c in ["onion", "potato", "tomato"]:
    evaluate_ensemble(c)